# **EEG-based Mental Workload Classification — Multi-Participant Analysis**

    Ana Muñoz Gil
    BSc in Biomedical Engineering, TFG
    May 2026

------

The objective of this code is to perform Spearman correlation and feature selection on processed EEG data of a dataset of 24 participants (LOPO) and evaluate the performance of five classifiers in predicting both obkective (N-back) and subjective (Bedford) mental workload labels from EEG data


### Settings

In [ ]:
import warnings
from sklearn.exceptions import SkipTestWarning
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

In [ ]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    LeaveOneGroupOut, GroupKFold, StratifiedKFold, GridSearchCV, cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score, f1_score
)
from scipy.stats import spearmanr
from IPython.display import display

PALETTE = ["#440154", "#30678D", "#64C5EB", "#FEB326", "#35B779"]
sns.set_theme(style="whitegrid", font_scale=1.1)

Paths for inputs and outputs 

In [ ]:
project_root     = "C://Users//Usuario//Desktop//UNI//TFG//python"
output_path      = os.path.join(project_root, "output_segments//FinalResults")

## 1. Load the data

Load the processed EEG data for all participants and merge into a unique dataset with all data for the entire cohort

In [ ]:
participant_ids  = [f"participant_{i:02d}" for i in range(1, 25)]
EXCLUDE          = []

features_list, markers_list, skipped = [], [], []

for pid in participant_ids:
    if pid in EXCLUDE:
        print(f"⏭️  {pid} excluido"); continue

    feat_path    = f"{project_root}//output_segments//{pid}//features//features_eeg_all.csv"
    markers_path = f"{project_root}//participantes//{pid}//psychopymarkers_data.csv"

    if not os.path.exists(feat_path) or not os.path.exists(markers_path):
        print(f"⚠️  {pid} — archivo no encontrado, saltando")
        skipped.append(pid); continue

    feat_df    = pd.read_csv(feat_path)
    markers_df = pd.read_csv(markers_path)
    feat_df['participant_id']    = pid
    markers_df['participant_id'] = pid
    features_list.append(feat_df)
    markers_list.append(markers_df)
    print(f"✅ {pid} — {feat_df['epoch_index'].nunique()} epochs, "
          f"{feat_df['Channel'].nunique()} canales")

features_all = pd.concat(features_list, ignore_index=True)
markers_all  = pd.concat(markers_list,  ignore_index=True)


## 2. Exclude bad channels

Me lo pueod saltar (ya los expluyo en el preprocessing)

In [ ]:
BAD_CHANNELS = ['chann_20', 'chann_24']
features_all = features_all[~features_all['Channel'].isin(BAD_CHANNELS)].copy()

## 3. Map channels to respective brain region

Each channel can be asigned to a specific brain region, map channels to regions and aggregate each signal by region to get a more explainable datasets

In [ ]:
REGION_MAP = {
    'chann_1': 'Frontal', 'chann_2': 'Frontal', 'chann_3': 'Frontal',
    'chann_4': 'Frontal', 'chann_5': 'Frontal', 'chann_6': 'Frontal',
    'chann_7': 'Frontal', 'chann_21': 'Frontal', 'chann_22': 'Frontal',
    'chann_23': 'Frontal', 'chann_8': 'Temporal_L', 'chann_12': 'Temporal_R',
    'chann_9': 'Central', 'chann_10': 'Central', 'chann_11': 'Central',
    'chann_13': 'Parietal', 'chann_14': 'Parietal', 'chann_15': 'Parietal',
    'chann_16': 'Parietal', 'chann_17': 'Parietal',
    'chann_18': 'Occipital', 'chann_19': 'Occipital',
}
REGIONS   = sorted(set(REGION_MAP.values()))
FEAT_BASE = ['Delta_rel', 'Theta_rel', 'Alpha_rel', 'Beta_rel', 'Gamma_rel',
             'ratio_theta_alpha', 'ratio_theta_beta', 'ratio_beta_alpha_theta']

For each set of  (participant, epoch, difficulty), mean of the channels in each region

In [ ]:
region_agg_rows = []
for pid in sorted(features_all['participant_id'].unique()): # recorro cada participante
    df_pid = features_all[features_all['participant_id'] == pid] # recorro las filas de un participante
    for (ep_idx, diff), grp in df_pid.groupby(['epoch_index', 'difficulty']):  # recorro cada epoch (agrupo id epoch con su etiqueta de difficulty)
        row = {'participant_id': pid, 'epoch_index': ep_idx, 'difficulty': diff}
        for region in REGIONS:
            channels_in_region = [ch for ch, reg in REGION_MAP.items() if reg == region] #busca en REGION_MAP que canales pertenecen a la region "reg"
            grp_region = grp[grp['Channel'].isin(channels_in_region)] #grp_region filtra grp para quedarme solo con las filas de esos canales
            for feat in FEAT_BASE:
                if feat in grp_region.columns:
                    row[f"{feat}_{region}"] = grp_region[feat].mean()#calcula la media de la feature "feat" de los canales de la region
        region_agg_rows.append(row) #junta todas las filas (una por epoch de cada participante) en el dataframe final_

df_regions = pd.DataFrame(region_agg_rows)

In [ ]:
print(f"Dataset por región: {df_regions.shape[0]} epochs × {df_regions.shape[1]} columnas")
print(f"Columnas EEG: {[c for c in df_regions.columns if c not in ['participant_id','epoch_index','difficulty']]}")

In [ ]:
#Heatmap: media de cada feature por dificultad (media de los pacientes)

region_feat_cols = [c for c in df_regions.columns
                    if c not in ['participant_id', 'epoch_index', 'difficulty']]

feat_means = df_regions.groupby('difficulty')[region_feat_cols].mean()

plt.figure(figsize=(12, 12))
sns.heatmap(feat_means.T, cmap='viridis', annot=True, fmt='.3f')
plt.title('Media de features EEG por dificultad — 24 participantes (por región)')
plt.tight_layout()
os.makedirs(f"{output_path}", exist_ok=True)
plt.savefig(f"{output_path}/01_features_global_heatmap.png",
            dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Guardado: 01_features_global_heatmap.png")

EEG signal in each region by patient: signal of region at all epochs

In [ ]:
def plot_epochs_by_difficulty(df_regions, participant_id, figsize=(18, 5)):

    import matplotlib.pyplot as plt
    import numpy as np

    DIFF_ORDER  = ['1_BACK', '2_BACK', '3_BACK']
    DIFF_COLORS = {'1_BACK': '#d4eac8', '2_BACK': '#fde8c8', '3_BACK': '#f5c6c6'}
    FEATS_BASE  = ['Delta_rel', 'Theta_rel', 'Alpha_rel', 'Beta_rel', 'Gamma_rel']
    
    df_p = df_regions[df_regions['participant_id'] == participant_id].copy()
    df_p['diff_order'] = df_p['difficulty'].map(
        {d: i for i, d in enumerate(DIFF_ORDER)}
    )
    df_p = df_p.sort_values(['diff_order', 'epoch_index']).reset_index(drop=True)
    df_p['plot_x'] = np.arange(len(df_p))

    regions     = sorted(set(REGION_MAP.values()))
    line_colors = plt.cm.tab10.colors
    linestyles  = ['-', '--', '-.', ':', (0,(3,1,1,1))]

    n_regions = len(regions)
    fig, axes = plt.subplots(
        n_regions, 1,
        figsize=(figsize[0], figsize[1] * n_regions),
        sharex=True
    )
    fig.patch.set_facecolor('white')

    for ax, region in zip(axes, regions):
        feats = [f"{fb}_{region}" for fb in FEATS_BASE
                 if f"{fb}_{region}" in df_p.columns]

        ax.set_facecolor('white')
        ax.grid(False)

        for diff in DIFF_ORDER:
            idx = df_p[df_p['difficulty'] == diff]['plot_x']
            if len(idx) == 0:
                continue
            x0, x1 = idx.min() - 0.5, idx.max() + 0.5
            ax.axvspan(x0, x1, color=DIFF_COLORS[diff], alpha=0.35, zorder=0)
            if diff != DIFF_ORDER[-1]:
                ax.axvline(x1, color='#888888', linewidth=1.2,
                           linestyle='--', alpha=0.7, zorder=1)
            ax.text((x0 + x1) / 2, 1.01, diff,
                    transform=ax.get_xaxis_transform(),
                    ha='center', va='bottom', fontsize=9,
                    color='#555555', style='italic')

        for i, feat in enumerate(feats):
            ax.plot(
                df_p['plot_x'], df_p[feat],
                color=line_colors[i % len(line_colors)],
                linestyle=linestyles[i % len(linestyles)],
                linewidth=1.5,
                marker='o', markersize=2.5,
                label=feat.replace(f"_{region}", ""),
                zorder=2,
            )

        ax.set_ylabel('Relative Power - Proportion', fontsize=9)
        ax.set_title(region, fontsize=10, loc='left', pad=4)
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
        ax.spines[['top', 'right']].set_visible(False)

    # eje X solo en el último subplot
    step = 10
    tick_pos = df_p['plot_x'][::step].values
    tick_lbl = df_p['epoch_index'][::step].values
    axes[-1].set_xticks(tick_pos)
    axes[-1].set_xticklabels(tick_lbl, rotation=45, fontsize=8)
    axes[-1].set_xlabel('Epoch (ordenado por difficulty)')

    fig.suptitle(f' {participant_id}', fontsize=13, y=1.001)
    plt.tight_layout()

    os.makedirs(
        f"{output_path}/regions_by_participant", exist_ok=True
    )
    out_path = (f"{output_path}/regions_by_participant/{participant_id}.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close()
    print(f"Guardado: {out_path}")

for pid in  sorted(df_regions['participant_id'].unique()):
    plot_epochs_by_difficulty(df_regions, participant_id=pid)

## 4. Assign Bedford label to segment epochs

In the end of every segment, a bedford scale value is given by the participant. We will asign this label to the entire segment

In [ ]:
## Functions to assign bedford scale and labels

# # sacamos el correct_ratio y el bedford_score de cada segmento de cada participante
def extract_labels_from_markers(markers_df):
    markers_df = markers_df.sort_values('timestamp').reset_index(drop=True)
    rows, seg_idx, i = [], 0, 0
    while i < len(markers_df):
        marker = str(markers_df.iloc[i]['marker'])
        if 'BACK_START' in marker:
            match = re.search(r'(\d)_BACK_START', marker)
            if not match: i += 1; continue#si por lo que sea no hubiera digito, pasamos a la siguiente linea
            n_back = int(match.group(1)) #coge el digito de match y lo hace integer
            j, correct, wrong = i + 1, 0, 0
            while j < len(markers_df):
                m = str(markers_df.iloc[j]['marker'])
                if 'BACK_END' in m: j += 1; break
                elif m == 'Correct': correct += 1
                elif 'Wrong' in m:   wrong   += 1
                j += 1 #con esto calculamos todos los correct y wrong de cada bloque
            total         = correct + wrong
            correct_ratio = correct / total if total > 0 else np.nan
            bedford_score = np.nan
            for k in range(j, min(j + 10, len(markers_df))):
                bm = str(markers_df.iloc[k]['marker'])
                bed_match = re.search(r'(\d)_BEDFORD_(\d+)', bm)
                if bed_match and int(bed_match.group(1)) == n_back:
                    bedford_score = int(bed_match.group(2)); break
            rows.append({'segment_idx': seg_idx, 'n_back': n_back,
                         'correct_ratio': correct_ratio, 'bedford_score': bedford_score})
            seg_idx += 1; i = j
        else:
            i += 1
    return pd.DataFrame(rows)

def bedford_to_label(score):
    """LOW=1-2 | MEDIUM=3-4 (excluido) | HIGH=5-10"""
    if pd.isna(score): return np.nan
    if score <= 2: return 'LOW'
    elif score <= 5: return 'MEDIUM'
    else: return 'HIGH'


Assign bedford scores to segments (epochs) and classify in HIGH, LOW or MEDIUM!

In [ ]:
reps_per_diff = {'1_BACK': 3, '2_BACK': 3, '3_BACK': 3}
all_labeled   = []
for pid in sorted(df_regions['participant_id'].unique()):
    markers_df = markers_all[markers_all['participant_id'] == pid].copy() #markers de un pax
    seg_df_pid = extract_labels_from_markers(markers_df)
    df_pid     = df_regions[df_regions['participant_id'] == pid].copy()#las features por region del pax
    df_pid[['segment_idx', 'bedford_score', 'correct_ratio']] = np.nan
    for diff, n_reps in reps_per_diff.items():
        n_back = int(diff[0])
        mask   = df_pid['difficulty'] == diff
        ep_ids = np.sort(df_pid.loc[mask, 'epoch_index'].unique())
        splits = np.array_split(ep_ids, n_reps) #coge todas las epochs del segmento N
        segs   = seg_df_pid[seg_df_pid['n_back'] == n_back].reset_index(drop=True)
        for split_idx, split_ep in enumerate(splits):
            if split_idx >= len(segs): continue
            srow = segs.iloc[split_idx]
            em   = mask & df_pid['epoch_index'].isin(split_ep)
            df_pid.loc[em, 'segment_idx']   = srow['segment_idx']
            df_pid.loc[em, 'bedford_score'] = srow['bedford_score']
            df_pid.loc[em, 'correct_ratio'] = srow['correct_ratio']
    all_labeled.append(df_pid)

df_all = pd.concat(all_labeled, ignore_index=True)
df_all['n_back']        = df_all['difficulty'].str.extract(r'(\d)').astype(int)
df_all['bedford_score'] = pd.to_numeric(df_all['bedford_score'], errors='coerce')
df_all['bedford_label'] = df_all['bedford_score'].apply(bedford_to_label)
bedford_order             = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2}
df_all['bedford_encoded'] = df_all['bedford_label'].map(bedford_order)

# df_all_bedford: solo LOW (<=2) y HIGH (>=5), para clasificadores Bedford
df_all_bedford = df_all[df_all['bedford_label'].isin(['LOW', 'HIGH'])].copy()
df_all_bedford['bedford_encoded'] = df_all_bedford['bedford_label'].map({'LOW': 0, 'HIGH': 1})

print(f"Dataset completo : {df_all.shape[0]} epochs × {df_all.shape[1]} columnas")
print(f"Dataset Bedford  : {df_all_bedford.shape[0]} epochs (LOW={( df_all_bedford['bedford_label']=='LOW').sum()}, HIGH={(df_all_bedford['bedford_label']=='HIGH').sum()})")


In [ ]:
## Lets se
print(f"\nDistribución Bedford:")
print(df_all['bedford_label'].value_counts().reindex(['LOW', 'MEDIUM', 'HIGH']))
print(f"\nDistribución N-back:")
print(df_all['n_back'].value_counts().sort_index())

The Bedford Label classes are more or less balanced, with the MEDIUM class being the most represented class. As expected the N-back classes are balanced as each task was separated in epochs

There are multiple options in which we could classify the bedford scale to be low, medium or high. After doing the confusion matrix with bedford scores and n-back task, we decided that LOW (0-2), MEDIUM (3-5) and HIGH (6-10) is the most representative classification for mental workload labels

In [ ]:
df_all_changes = df_all.copy()
df_cm = (df_all_changes[["participant_id", "segment_idx", "difficulty", "bedford_label"]].drop_duplicates())

df_cm["bedford_label"] = pd.Categorical( df_cm["bedford_label"],
    categories=["LOW", "MEDIUM", "HIGH"],
    ordered=True)

df_cm["difficulty"] = pd.Categorical( df_cm["difficulty"],
    categories=["1_BACK", "2_BACK", "3_BACK"],
    ordered=True)

cm = pd.crosstab(df_cm["difficulty"], df_cm["bedford_label"])
cm = cm.reindex(index=["1_BACK", "2_BACK", "3_BACK"], columns=["LOW", "MEDIUM","HIGH"])

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Mental Workload (Bedford label)")
plt.ylabel("Difficulty")
#plt.title("N-back level vs Subjective mental workload")
plt.show()

## 5. Explore the dataset

In [ ]:
signals_to_visualize = ['Theta_rel', 'Alpha_rel', 'ratio_theta_alpha']

In [ ]:
palette = {
    '1_BACK': '#d4eac8',
    '2_BACK': '#fde8c8',
    '3_BACK': '#f5c6c6'
}

for feat in signals_to_visualize:

    feat_cols = [f"{feat}_{r}" for r in REGIONS if f"{feat}_{r}" in df_regions.columns]

    fig, axes = plt.subplots( int(np.ceil(len(feat_cols)/3)), 3, figsize=(15, 4*int(np.ceil(len(feat_cols)/3))))

    axes = axes.flatten()
    for ax, col in zip(axes, feat_cols):
        tmp = (
            df_regions
            .groupby(['participant_id','difficulty'])[col]
            .mean()
            .reset_index()
            .rename(columns={col:'value'})
        )
        # Violin plot allows to see the distribtion of the data
        sns.violinplot(
            data=tmp,
            x='difficulty',
            y='value',
            palette=palette,
            inner='box',
            cut=0,
            ax=ax
        )

        sns.stripplot(
            data=tmp,
            x='difficulty',
            y='value',
            color='black',
            size=4,
            alpha=0.7,
            jitter=0.15,
            ax=ax
        )
        ax.set_title(col.replace(f"{feat}_", ""))
        ax.set_xlabel("")
        ax.set_ylabel("")

    for ax in axes[len(feat_cols):]:
        fig.delaxes(ax)

    plt.suptitle(f'{feat}: Mean value per participant by difficulty and region')
    plt.tight_layout()

    os.makedirs( f"{output_path}/Signal_distributions", exist_ok=True )
    fname = f"02_violin_{feat}.png"
    plt.savefig(f"{output_path}/Signal_distributions/{fname}", dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    print(f"Guardado: {fname}")

Violines solo para theta frontal y para ratio theta/alpha frontal (son los que mas info aportan, plot de alpha o plot de todos podria ir en anexos)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

palette = {
    '1_BACK': '#d4eac8',
    '2_BACK': '#fde8c8',
    '3_BACK': '#f5c6c6'
}

features_to_plot = [
    ('Theta_rel', 'Frontal', 'Relative Theta Power'),
    ('ratio_theta_alpha', 'Frontal', 'Theta/Alpha Ratio')
]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, (feat, region, label) in zip(axes, features_to_plot):
    col = f"{feat}_{region}"
    
    tmp = (
        df_regions
        .groupby(['participant_id', 'difficulty'])[col]
        .mean()
        .reset_index()
        .rename(columns={col: 'value'})
    )
    
    sns.violinplot(
        data=tmp,
        x='difficulty',
        y='value',
        palette=palette,
        inner='box',
        cut=0,
        ax=ax,
        order=['1_BACK', '2_BACK', '3_BACK']
    )
    sns.stripplot(
        data=tmp,
        x='difficulty',
        y='value',
        color='black',
        size=5,
        alpha=0.7,
        jitter=0.12,
        ax=ax,
        order=['1_BACK', '2_BACK', '3_BACK']
    )
    
    ax.set_title(f'Frontal {label}', fontsize=13, fontweight='bold')
    ax.set_xlabel('N-back level', fontsize=11)
    ax.set_ylabel(label, fontsize=11)
    ax.set_xticklabels(['1-back', '2-back', '3-back'], fontsize=10)

plt.suptitle(
    'Frontal EEG features across N-back difficulty levels\n(each dot = one participant mean)',
    fontsize=13, y=1.02
)
plt.tight_layout()

os.makedirs(f"{output_path}/Signal_distributions", exist_ok=True)
plt.savefig(
    f"{output_path}/Signal_distributions/02_violin_frontal_theta_ratio.png",
    dpi=150, bbox_inches='tight'
)
plt.show()
plt.close()

In [ ]:
# --- 2.3 Heatmap por banda × región × dificultad ---
for feat in ['Theta_rel', 'Alpha_rel', 'ratio_theta_alpha', 'ratio_theta_beta']:
    feat_cols = [f"{feat}_{r}" for r in REGIONS if f"{feat}_{r}" in df_regions.columns]
    band_means = df_regions.groupby('difficulty')[feat_cols].mean()
    band_means.columns = [c.replace(f"{feat}_", "") for c in band_means.columns]

    plt.figure(figsize=(10, 4))
    sns.heatmap(band_means.T, cmap='viridis', annot=True, fmt='.3f')
    plt.title(f'Mean {feat} for N-back level x region — 24 participants')
    plt.xlabel('N-back level')
    plt.ylabel('Region')
    plt.tight_layout()
    fname = f"02_heatmap_{feat}.png"
    plt.savefig(f"{output_path}/{fname}",
                dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"Guardado: {fname}")

# --- 2.4 Verificación ---
print(f"\n✅ Regiones: {REGIONS}")
print(f"✅ Participantes: {df_regions['participant_id'].nunique()}")
print(f"✅ Epochs totales: {df_regions.shape[0]}")

### 5.2 Bedford scale vs difficulty correlation (Spearman's rank test)

In [ ]:
from scipy.stats import spearmanr

REGION_NAMES = ['Central', 'Frontal', 'Occipital', 'Parietal', 'Temporal_R', 'Temporal_L']
FEAT_BASE    = ['Delta_rel', 'Theta_rel', 'Alpha_rel', 'Beta_rel', 'Gamma_rel',
                'ratio_theta_alpha', 'ratio_theta_beta', 'ratio_beta_alpha_theta']

# output dir
spearman_dir = f"{output_path}/Spearman_correlation_test"
os.makedirs(spearman_dir, exist_ok=True)

# --- 4.1 Promediar por bloque (participant × segment_idx) ---
region_feat_cols = [c for c in df_all.columns
                    if any(c.startswith(f + '_') for f in FEAT_BASE)] #coge todas las features

block_cols = region_feat_cols + ['correct_ratio'] #añado correct_ratio a las features
df_blocks  = (df_all
              .groupby(['participant_id', 'segment_idx', 'n_back', 'bedford_score'])
              [block_cols]
              .mean()
              .reset_index()) #promedio todas las epochs de cada bloque (nos quedamos con 9 bloques por sujeto para hacer el Spearman)

print(f"Bloques totales: {len(df_blocks)} (esperado: 216)") # 9 sujetos * 24 bloques = 216
participants = sorted(df_blocks['participant_id'].unique())

# =============================================================================
# CASO 1: EEG vs N-back — 8 tablas (una por feature)
# =============================================================================
print("\n" + "=" * 50)
print("CASO 1: EEG vs N-back")
print("=" * 50)
# para cada sujeto y cada region, correlaciono la feature EEG con el nivel n-back
# respondo a: ¿El EEG sigue la dificulad impuesta?
for feat in FEAT_BASE:
    rows = []
    for pid in participants:
        df_pid = df_blocks[df_blocks['participant_id'] == pid]
        row    = {'participant_id': pid}
        for region in REGION_NAMES:
            col = f"{feat}_{region}"
            if col not in df_pid.columns:
                row[f"rho_{region}"] = np.nan
                row[f"p_{region}"]   = np.nan
                continue
            mask = ~(df_pid[col].isna() | df_pid['n_back'].isna())
            if mask.sum() < 3:  #saltamos a sujetos con menos de 3 puntos validos (proteccion)
                row[f"rho_{region}"] = np.nan
                row[f"p_{region}"]   = np.nan 
                continue 
            rho, pval = spearmanr(df_pid.loc[mask, col], df_pid.loc[mask, 'n_back'])
            row[f"rho_{region}"] = round(rho,  4)
            row[f"p_{region}"]   = round(pval, 4)
        rows.append(row)

    df_table = pd.DataFrame(rows)

    # Ordenar columnas: rho_Central, p_Central, rho_Frontal, p_Frontal, ...
    col_order = ['participant_id']
    for region in REGION_NAMES:
        col_order += [f"rho_{region}", f"p_{region}"]
    df_table = df_table[col_order]

    # Guardar
    fname = f"{spearman_dir}/EEG_vs_Nback_{feat}.csv"
    df_table.to_csv(fname, index=False)

    # Printear resumen
    print(f"\n  → {feat}:")
    print(df_table.to_string(index=False))
    print(f"  Median rho for region:")
    for region in REGION_NAMES:
        med = df_table[f"rho_{region}"].median() #bien que se use la mediana en vez de la media (con solo 9 puntos por sujeto, algunos rhos pueden ser extremos y la meidana es mas robusta a esos outliers)
        pct = (df_table[f"p_{region}"] < 0.05).mean() * 100
        print(f"    {region}: ρ={med:.3f}, {pct:.0f}% significatives") 

print(f"\n✅ Guardadas 8 tablas EEG vs N-back en: {spearman_dir}")

# =============================================================================
# CASO 2: EEG vs Bedford — 8 tablas (una por feature)
# =============================================================================
print("\n" + "=" * 50)
print("CASO 2: EEG vs Bedford")
print("=" * 50)

# ¿objetivo y subjetivo se correlacionan? ¿EEG sigue la carga percibida?
for feat in FEAT_BASE:
    rows = []
    for pid in participants:
        df_pid = df_blocks[df_blocks['participant_id'] == pid]
        row    = {'participant_id': pid}
        for region in REGION_NAMES:
            col = f"{feat}_{region}"
            if col not in df_pid.columns:
                row[f"rho_{region}"] = np.nan
                row[f"p_{region}"]   = np.nan
                continue
            mask = ~(df_pid[col].isna() | df_pid['bedford_score'].isna())
            if mask.sum() < 3:
                row[f"rho_{region}"] = np.nan
                row[f"p_{region}"]   = np.nan
                continue
            rho, pval = spearmanr(df_pid.loc[mask, col],
                                  df_pid.loc[mask, 'bedford_score'])
            row[f"rho_{region}"] = round(rho,  4)
            row[f"p_{region}"]   = round(pval, 4)
        rows.append(row)

    # ordenar
    df_table = pd.DataFrame(rows)
    col_order = ['participant_id']
    for region in REGION_NAMES:
        col_order += [f"rho_{region}", f"p_{region}"]
    df_table = df_table[col_order]
    
    # guardar
    fname = f"{spearman_dir}/EEG_vs_Bedford_{feat}.csv"
    df_table.to_csv(fname, index=False)

    # print summary
    print(f"\n  → {feat}:")
    print(df_table.to_string(index=False))
    print(f"  Mediana rho por región:")
    for region in REGION_NAMES:
        med = df_table[f"rho_{region}"].median()
        pct = (df_table[f"p_{region}"] < 0.05).mean() * 100
        print(f"    {region}: ρ={med:.3f}, {pct:.0f}% significatives")

print(f"\n✅ Guardadas 8 tablas EEG vs Bedford en: {spearman_dir}")

# =============================================================================
# CASO 3: Bedford vs N-back — 1 tabla
# =============================================================================
print("\n" + "=" * 50)
print("CASO 3: Bedford vs N-back")
print("=" * 50)

# correlaciono n-abck con bedford.
# el sujeto percibe la carga real?
rows = []
for pid in participants:
    df_pid = df_blocks[df_blocks['participant_id'] == pid]
    mask   = ~(df_pid['bedford_score'].isna() | df_pid['n_back'].isna())
    if mask.sum() < 3:
        rows.append({'participant_id': pid, 'rho': np.nan, 'p-value': np.nan, 'Significant': np.nan})
        continue
    rho, pval = spearmanr(df_pid.loc[mask, 'bedford_score'],
                          df_pid.loc[mask, 'n_back'])
    rows.append({
        'participant_id': pid,
        'rho':            round(rho,  4),
        'p-value':        round(pval, 4),
        'Significant':    pval < 0.05,
    })

df_bed_nback = pd.DataFrame(rows)
print(df_bed_nback.to_string(index=False))
print(f"\nMediana rho:        {df_bed_nback['rho'].median():.4f}")
print(f"Media rho:          {df_bed_nback['rho'].mean():.4f}")
print(f"Std rho:            {df_bed_nback['rho'].std():.4f}")
print(f"% significativos:   {df_bed_nback['Significant'].mean()*100:.1f}%")

fname = f"{spearman_dir}/Bedford_vs_Nback.csv"
df_bed_nback.to_csv(fname, index=False)
print(f"\n✅ Guardada tabla Bedford vs N-back en: {spearman_dir}")

# =============================================================================
# HEATMAPS RESUMEN
# =============================================================================
print("\nGenerando heatmaps resumen...")

for target, prefix in [('N-back', 'EEG_vs_Nback'), ('Bedford', 'EEG_vs_Bedford')]:
    for feat in FEAT_BASE:
        fname_csv = f"{spearman_dir}/{prefix}_{feat}.csv"
        df_t      = pd.read_csv(fname_csv)

        rho_cols = [f"rho_{r}" for r in REGION_NAMES]
        p_cols   = [f"p_{r}"   for r in REGION_NAMES]

        rho_matrix = df_t.set_index('participant_id')[rho_cols]
        rho_matrix.columns = REGION_NAMES
        p_matrix   = df_t.set_index('participant_id')[p_cols]
        p_matrix.columns   = REGION_NAMES

        # Anotaciones: rho con * si p<0.05
        annots = rho_matrix.copy().astype(str)
        for r in REGION_NAMES:
            for pid in rho_matrix.index:
                rho_v = rho_matrix.loc[pid, r]
                p_v   = p_matrix.loc[pid, r]
                sig   = '*' if p_v < 0.05 else ''
                annots.loc[pid, r] = f"{rho_v:.2f}{sig}"

        plt.figure(figsize=(10, 10))
        sns.heatmap(rho_matrix.astype(float), cmap='coolwarm', center=0,
                    vmin=-1, vmax=1, annot=annots, fmt='',
                    linewidths=0.5)
        plt.title(f'{feat}\nEEG vs {target} — ρ por participante × región\n(* = p<0.05)')
        plt.xlabel('Región cerebral')
        plt.ylabel('Participante')
        plt.tight_layout()
        img_fname = f"{spearman_dir}/{prefix}_{feat}_heatmap.png"
        plt.savefig(img_fname, dpi=150, bbox_inches='tight')
        plt.close()

print(f"✅ Heatmaps guardados en: {spearman_dir}")

# --- Heatmap Bedford vs N-back ---
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
colors_bar = [PALETTE[1] if r > 0 else PALETTE[0] for r in df_bed_nback['rho']]
axes[0].barh(df_bed_nback['participant_id'], df_bed_nback['rho'],
             color=colors_bar, edgecolor='black')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('ρ Bedford vs N-back\n(¿Perciben la dificultad?)')
axes[0].set_xlabel('ρ Spearman')

colors_p = [PALETTE[4] if p < 0.05 else PALETTE[2] for p in df_bed_nback['p-value']]
axes[1].barh(df_bed_nback['participant_id'], df_bed_nback['p-value'],
             color=colors_p, edgecolor='black')
axes[1].axvline(0.05, color='red', linestyle='--', label='p=0.05')
axes[1].set_title('p-value Bedford vs N-back\n(verde = significativo)')
axes[1].set_xlabel('p-value')
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{spearman_dir}/Bedford_vs_Nback_plot.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Guardado: Bedford_vs_Nback_plot.png")

coeficiente rho correlaciones

In [ ]:
# =============================================================================
# RESUMEN: TOP FEATURES POR RHO
# =============================================================================
print("\n" + "=" * 70)
print("TOP FEATURES POR RHO MEDIANO")
print("=" * 70)

summary_rows = []

for target, prefix in [('N-back', 'EEG_vs_Nback'), ('Bedford', 'EEG_vs_Bedford')]:
    for feat in FEAT_BASE:
        fname_csv = f"{spearman_dir}/{prefix}_{feat}.csv"
        df_t      = pd.read_csv(fname_csv)

        for region in REGION_NAMES:
            rho_col = f"rho_{region}"
            p_col   = f"p_{region}"
            if rho_col not in df_t.columns:
                continue
            median_rho = df_t[rho_col].median()
            mean_rho   = df_t[rho_col].mean()
            std_rho    = df_t[rho_col].std()
            pct_sig    = (df_t[p_col] < 0.05).mean() * 100

            summary_rows.append({
                'Target':          target,
                'Feature':         feat,
                'Region':          region,
                'Feature_Region':  f"{feat}_{region}",
                'median_rho':      round(median_rho, 4),
                'mean_rho':        round(mean_rho,   4),
                'std_rho':         round(std_rho,    4),
                'abs_median_rho':  round(abs(median_rho), 4),
                'pct_significant': round(pct_sig, 1),
            })

df_summary = pd.DataFrame(summary_rows)

# --- Print top 10 por target ---
for target in ['N-back', 'Bedford']:
    print(f"\n🏆 Top 10 features (|ρ| mediano) — EEG vs {target}:")
    top10 = (df_summary[df_summary['Target'] == target]
             .sort_values('abs_median_rho', ascending=False)
             .head(10)[['Feature_Region', 'median_rho', 'std_rho', 'pct_significant']])
    print(top10.to_string(index=False))

# --- Guardar CSV resumen ---
df_summary.to_csv(f"{spearman_dir}/summary_top_features.csv", index=False)
print(f"\n✅ Guardado: summary_top_features.csv")

# --- Guardar Excel con múltiples hojas ---
excel_path = f"{spearman_dir}/spearman_summary.xlsx"
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

    # Hoja 1: Top features N-back
    (df_summary[df_summary['Target'] == 'N-back']
     .sort_values('abs_median_rho', ascending=False)
     [['Feature_Region', 'Feature', 'Region', 'median_rho', 'std_rho', 'pct_significant']]
     .to_excel(writer, sheet_name='Top_Nback', index=False))

    # Hoja 2: Top features Bedford
    (df_summary[df_summary['Target'] == 'Bedford']
     .sort_values('abs_median_rho', ascending=False)
     [['Feature_Region', 'Feature', 'Region', 'median_rho', 'std_rho', 'pct_significant']]
     .to_excel(writer, sheet_name='Top_Bedford', index=False))

    # Hoja 3: Bedford vs N-back
    df_bed_nback.to_excel(writer, sheet_name='Bedford_vs_Nback', index=False)

    # Hoja 4: Resumen completo
    df_summary.to_excel(writer, sheet_name='Completo', index=False)

print(f"✅ Guardado Excel: spearman_summary.xlsx")

# --- Bar plot top 15 ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, target in zip(axes, ['N-back', 'Bedford']):
    top15 = (df_summary[df_summary['Target'] == target]
             .sort_values('abs_median_rho', ascending=False)
             .head(15))
    colors = [PALETTE[1] if r > 0 else PALETTE[0] for r in top15['median_rho']]
    ax.barh(top15['Feature_Region'], top15['median_rho'],
            color=colors, edgecolor='black')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.axvline(0.30, color='red', linestyle='--', alpha=0.5, label='ρ=0.30')
    ax.axvline(-0.30, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'Top 15 features — EEG vs {target}\n(ordenadas por |ρ| mediano)')
    ax.set_xlabel('ρ mediano Spearman')
    ax.invert_yaxis()
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{spearman_dir}/top_features_barplot.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("Guardado: top_features_barplot.png")

In [ ]:
print("Distribución de clases Bedford:")
print(df_all['bedford_label'].value_counts())
print("\nPor participante:")
print(df_all.groupby('participant_id')['bedford_label'].value_counts().unstack(fill_value=0))

hay mucha variabilidad entre sujetos (unos nunca marcan high, otros nunca marcan ni medium ni high, otros marcan casi todo medium ...)

## 6. Classification

In [ ]:

def get_selector(method, k):
    if method == 'anova':
        return SelectKBest(f_classif, k=k)
    elif method == 'mutual_info':
        score_func = lambda X, y: mutual_info_classif(X, y, random_state=42)
        return SelectKBest(score_func, k=k)
    raise ValueError("method debe ser 'anova' o 'mutual_info'")


def save_confusion_matrix(y_true, y_pred, classes, title, filepath):
    """Guarda la confusion matrix de un modelo como PNG individual."""
    fig, ax = plt.subplots(figsize=(5, 4))
    cm      = confusion_matrix(y_true, y_pred, labels=classes)
    cm_pct  = (cm.T / cm.sum(axis=1)).T * 100
    disp    = ConfusionMatrixDisplay(confusion_matrix=cm,
                                     display_labels=classes)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i + 0.28, f"{cm_pct[i, j]:.1f}%",
                    ha="center", va="center", fontsize=9, color="gray")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  💾 Confusion matrix → {filepath}")


def plot_all_rocs(roc_data, experiment_name, filepath):
    """
    Dibuja en una sola figura las curvas ROC de todos los modelos de un experimento.
    roc_data: {clf_name: (fpr, tpr, auc_val)} o None si no disponible
    """
    fig, ax = plt.subplots(figsize=(6, 5))
    colors  = plt.cm.tab10.colors
    for i, (clf_name, data) in enumerate(roc_data.items()):
        if data is None:
            continue
        fpr, tpr, auc_val = data
        ax.plot(fpr, tpr, lw=2, color=colors[i % 10],
                label=f"{clf_name} (AUC={auc_val:.3f})")
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"ROC — {experiment_name}")
    ax.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  💾 ROC conjunta → {filepath}")


def save_confusion_matrix(y_true, y_pred, classes, title, filepath):
    """Guarda la confusion matrix de un modelo como PNG individual."""
    fig, ax = plt.subplots(figsize=(5, 4))
    # labels=range(len(classes)) para que coincida con y numérica,
    # display_labels=classes para mostrar los nombres en el eje
    cm      = confusion_matrix(y_true, y_pred)   # sin labels, infiere de y
    cm_pct  = (cm.T / cm.sum(axis=1)).T * 100
    disp    = ConfusionMatrixDisplay(confusion_matrix=cm,
                                     display_labels=classes)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i + 0.28, f"{cm_pct[i, j]:.1f}%",
                    ha="center", va="center", fontsize=9, color="gray")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  💾 Confusion matrix → {filepath}")
 

Now, lets define the classifiers that we are going to use and the hyperparameters of these models that we will perform tuning on

In [ ]:
# clf = classifier (initialize the model), param_grid = hyperparameter grid for GridSearchCV
def get_classifiers():
    return {
        "LDA": {
            "clf": LinearDiscriminantAnalysis(),
            "param_grid": {"clf__solver": ["svd"]},
        },
        "SVM": {
            "clf": SVC(class_weight="balanced", probability=True, random_state=42),
            "param_grid": {"clf__C": [0.1, 1, 10], "clf__kernel": ["rbf", "linear"]},
        },
        "Logistic Regression": {
            "clf": LogisticRegression(max_iter=1000, class_weight="balanced"),
            "param_grid": {"clf__C": [0.01, 0.1, 1, 10], "clf__solver": ["lbfgs"]},
        },
        "Random Forest": {
            "clf": RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1),
            "param_grid": {"clf__n_estimators": [100, 200], "clf__max_depth": [3, 5, 10]},
        },
        "Decision Tree": {
            "clf": DecisionTreeClassifier(class_weight="balanced", random_state=42),
            "param_grid": {"clf__max_depth": [3, 5, 10], "clf__min_samples_leaf": [2, 5, 10]},
        },
    }

### 6.1 Predict subjective mental workload (Bedford scale class)

Our first classification attempt is to see whether the classifiers are able to correctly predict the level of subjective metnal workload (Bedford label: LOW, MEDIUM, HIGH) based on EGG transformed features.

**FIRST** Prepare the data for classification. We need to separate the feature set and the label set (the true label for the model to predict). Since we are performing LOPO we will create as much models (of each type) as participants and report all metrics as the mean+-sd of the models. We will perform GridSearch Cross Validation for tunning the models and LOPO for testing (to control for biasing on the selection of the participant test set)

In [ ]:
def prepare_bedford_lopo(df_all):
    """
    Prepara datos para LOPO con clasificación BINARIA de umbrales FIJOS:
      - LOW  (0): bedford_score <= 2
      - HIGH (1): bedford_score >= 5
      - MEDIUM (3-4) ya excluido en df_all.
    Estandariza features por participante.
    """
    meta_cols = ["participant_id", "epoch_index", "difficulty", "segment_idx",
                 "bedford_score", "bedford_label", "bedford_encoded",
                 "correct_ratio", "n_back"]
    X_cols = [c for c in df_all.columns if c not in meta_cols]
    X_raw  = df_all[X_cols].values.astype(float)
    y      = df_all["bedford_encoded"].values
    groups = df_all["participant_id"].values

    for pid in df_all["participant_id"].unique():
        idx = df_all["participant_id"].values == pid
        X_raw[idx] = StandardScaler().fit_transform(X_raw[idx])

    nan_mask        = ~np.isnan(X_raw).any(axis=1)
    X_out, y_out, g = X_raw[nan_mask], y[nan_mask], groups[nan_mask]
    classes         = ['LOW (≤2)', 'HIGH (≥5)']

    print(f"  Bedford BINARIO (umbrales fijos) -> {nan_mask.sum()} epochs")
    print(f"  Distribucion: LOW={int((y_out==0).sum())}, HIGH={int((y_out==1).sum())}")

    one_class = []
    for pid in np.unique(g):
        yy = y_out[g == pid]
        if len(np.unique(yy)) < 2:
            one_class.append((pid, "solo HIGH" if yy[0] == 1 else "solo LOW", len(yy)))
    if one_class:
        print(f"  ⚠️ {len(one_class)} sujeto(s) con una sola clase (saltados en LOPO):")
        for pid, which, n in one_class:
            print(f"      {pid}: {which} ({n} epochs)")

    return X_out, y_out, g, X_cols, classes


In [ ]:
# NOTA: prepare_bedford_lopo_median reemplazada por umbrales fijos.
# Se mantiene como alias por si hay referencias en celdas posteriores.
prepare_bedford_lopo_median = prepare_bedford_lopo


In [ ]:
# separamos features (x) y labels (y). Sacamos los grupos para hacer nested LOPO
X_bed, y_bed, g_bed, cols_bed, bed_classes = prepare_bedford_lopo(df_all_bedford)

LOPO

In [ ]:

def nested_lopo_one_model(clf_cfg, X, y, groups, k_features, n_inner_splits=5):
    """
    Ejecuta nested LOPO CV para UN clasificador. ESTA ES LA FUNCION QUE ENTRENA EL MODELO Y PREDICE

    Devuelve:
        y_pred           : predicciones test (out-of-fold)
        y_score          : probabilidades test, shape (n_samples, n_classes) o None
        best_params_list : mejores hiperparámetros por fold
        train_accs       : balanced accuracy en TRAIN de cada fold
        val_accs         : balanced accuracy en validación interna (inner CV) de cada fold
    """
    # Generar todos los objetos vacios para almacenar resultados
    logo      = LeaveOneGroupOut() # LOPO cross validation
    n_classes = len(np.unique(y))
    y_pred    = np.empty_like(y) # aqui almacenamos todas las preddciones (de los 24 modelos de LOPO) --> vamos a calcular un test accuracy gloabl
    y_score   = np.full((len(y), n_classes), np.nan)
    best_params_list = [] # lista de los mejores valores para los hiperparametros (cada fold de LOPO puede tener uno mismo, pero nos quedamos con los mas seleccionados)
    train_accs, val_accs = [], [] # lista para guardar la accuracy del train y validation (inner CV) de cada fold de LOPO (24)
    test_accs = [] # lista para guardar la accuracy del test de cada fold de LOPO (24)

    # LOPO split (separa por paciente y entrena un modelo pcon todos los pacientes menos el, que lo usa para testear )
    for tr_idx, te_idx in logo.split(X, y, groups):
        X_tr, X_te = X[tr_idx], X[te_idx] # separamos en train y test
        y_tr       = y[tr_idx] # labels del trining set
        g_tr       = groups[tr_idx] # grupos del training set

        # Si el sujeto de test tiene UNA sola clase, accuracy de ese fold
        # seria degenerado -> lo saltamos (puede pasar tras binarizar). Se deja la
        # prediccion como -1 para no contaminar la matriz global (se filtra luego).
        if len(np.unique(y[te_idx])) < 2:
            y_pred[te_idx] = -1
            continue
        
        k    = min(k_features, X.shape[1])
        pipe = Pipeline([
            # ("scaler",   StandardScaler()),
            ("selector", SelectKBest(f_classif, k=k)),
            ("clf",      clf_cfg["clf"]),
        ]) # select k best features, and then train the classifier

        n_groups_tr  = len(np.unique(g_tr))
        inner_splits = min(n_inner_splits, n_groups_tr) # esto es un check pero siempre va a ser 5 (porque tenemos 24 sujetos, o sea que n_groups_tr es 23, asi que son 5-fold validation)

        if inner_splits >= 2 and clf_cfg["param_grid"]:
            # Nested CV: inner CV for hyperparameter tuning!! Aqui hacemos el grid search para encontrar los mejores hiperparametros del modelo (esta es la validacion)
            inner_cv  = GroupKFold(n_splits=inner_splits)
            cv_splits = list(inner_cv.split(X_tr, y_tr, g_tr)) # separa en 5 folds (train/val) para hacer el grid search
            # --- Grid Search para hyperparameter tuning ---
            grid = GridSearchCV(
                pipe, clf_cfg["param_grid"],
                cv=cv_splits,
                scoring="accuracy", n_jobs=-1, refit=True,
            )
            grid.fit(X_tr, y_tr) 
            model   = grid.best_estimator_      # selecciona el mejor modelo (con los mejores hiperparametros) de los 5 folds del inner CV
            val_acc = grid.best_score_          # selecciona la accuracy de ese mejor modelo (es la accuracy de validacion del inner CV)
            best_params_list.append(grid.best_params_)
        else:
            # Esto es por si no hay parámtros para hacer grid search (por ejemplo LDA no tiene hiperparametros, asi que no hace falta hacer el inner CV)
            pipe.fit(X_tr, y_tr)
            model   = pipe
            val_acc = np.nan
            best_params_list.append({})

        #### --- ya está elegido el mejor modelo con los mejores hiperparámetros ---- ###
        # Guarda el validation accuracy (del inner CV, es decir de la validacion interna) 
        val_accs.append(val_acc)
        # Train accuracy (haz predict con todo el dataset a ver si hay overfitting o no)
        train_accs.append(accuracy_score(y_tr, model.predict(X_tr)))
        
        # Ahora predice en el paciente que no has visto!! Este es el test accuracy (out-of-fold) que nos interesa para ver si el modelo generaliza a sujetos nuevos
        y_pred[te_idx] = model.predict(X_te) #saca las labels

        test_accs.append(   # predict!
            accuracy_score(
                y[te_idx],
                y_pred[te_idx]
            )
        )

        if hasattr(model, "predict_proba"):
            try:
                proba         = model.predict_proba(X_te)
                model_classes = model.named_steps["clf"].classes_
                for col, cls in enumerate(model_classes):
                    cls_idx = np.where(np.unique(y) == cls)[0][0]
                    y_score[te_idx, cls_idx] = proba[:, col]
            except Exception:
                pass

    if np.isnan(y_score).all():
        y_score = None

    ## Devolvemos todas las métricas y resultados del LOPO nested CV para este modelo
    # importante! devolvemos la test accuracy de cada fold (cada sujeto) para poder calcular la media y std de la test accuracy por sujeto. Pero tambien devolvemos las predicciones tal cual, que vamos a calcular la test accuracy gloabl
    return y_pred, y_score, best_params_list, train_accs, val_accs, test_accs


def run_lopo_per_model(X, y, groups, classes, experiment_name,
                       output_dir, k_features=30):
    """
    Evalúa cada clasificador por separado en LOPO nested CV. ESTA FUNCION LLAMA A LA ANTERIOR PARA CADA MODELO Y GUARDA TODOS LOS RESULTADOS JUNTITOS

    Guarda por modelo:
      - confusion matrix PNG
      - fpr/tpr para la ROC conjunta
      - métricas de train / val / test

    Al final guarda:
      - tabla resumen CSV + PNG
      - figura ROC conjunta PNG
    """
    # Crea e directorio de salida para guardar las figuras y resultados
    img_dir = os.path.join(output_dir, experiment_name.replace(" ", "_"))
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  LOPO NESTED CV (por modelo) — {experiment_name}")
    print(f"{'='*70}")

    # Aqui llama a todos los tipso de clasificadores!!!!!!!!
    classifiers  = get_classifiers()
    all_metrics  = {}   # Aqui guardamos las metricas de acuracy y etc
    roc_data     = {}   # Aqui guardamos la info para el ROC plot que vamos a hacer todos los modelos juntos
    fold_rows    = []   # resultados fold a fold

    # Ahroa, uno por uno recorremos todos los modelos y hacemos LOPO y predecimos 
    for clf_name, clf_cfg in classifiers.items():
        print(f"\n{'─'*60}")
        print(f"  ▶  {clf_name}")
        print(f"{'─'*60}")

        # Entrena,predice y saca los resultados!!!
        y_pred, y_score, best_params, train_accs, val_accs, test_accs = nested_lopo_one_model(
            clf_cfg, X, y, groups, k_features=k_features)

        # Filtrar sujetos saltados (test de una sola clase -> y_pred == -1) para que
        # NO contaminen las metricas globales ni la matriz de confusion.
        keep_mask = (y_pred != -1)
        n_skipped = int((~keep_mask).sum())
        if n_skipped:
            print(f"  (Se excluyen {n_skipped} epochs de sujetos de test con una sola clase)")
        y_eval     = y[keep_mask]
        y_pred_ev  = y_pred[keep_mask]
        y_score_ev = y_score[keep_mask] if y_score is not None else None

        # ── Métricas test ──────────────────────────────────────────────────────
        bal_acc_test = accuracy_score(y_eval, y_pred_ev)   # esto calcula toooooooodo el acuracy global
        report       = classification_report(y_eval, y_pred_ev, output_dict=True,
                                             zero_division=0,
                                             target_names=[str(c) for c in classes])
        macro        = report["macro avg"]

        # ── Métricas train / val ───────────────────────────────────────────────
        train_mean = np.nanmean(train_accs)
        train_std  = np.nanstd(train_accs)
        val_mean   = np.nanmean(val_accs)
        val_std    = np.nanstd(val_accs)
        test_mean = np.nanmean(test_accs)
        test_std  = np.nanstd(test_accs)

        # ── Filas fold a fold ──────────────────────────────────────────────────
        for fold_i, (tr_a, va_a, te_a) in enumerate(zip(train_accs, val_accs, test_accs)):
            fold_rows.append({
                "model":      clf_name,
                "fold":       fold_i,
                "train_bal_acc": round(tr_a, 4),
                "val_bal_acc":   round(va_a, 4) if not np.isnan(va_a) else None,
                "test_bal_acc":  round(te_a, 4),
            })

        print(f"\n  Hiperparámetros más frecuentes:")
        # Dime qué valores de los hiperparámetros se han elegido más veces como mejores
        if best_params and best_params[0]:
            param_keys = best_params[0].keys()
            for pk in param_keys:
                vals_pk = [bp[pk] for bp in best_params if pk in bp]
                from collections import Counter
                most_common = Counter(vals_pk).most_common(1)[0]
                print(f"    {pk}: {most_common[0]}  (en {most_common[1]}/{len(best_params)} folds)")
        else:
            print("    (sin grid search para este modelo)")

        print(f"\n  Train  Balanced Accuracy : {train_mean:.4f} ± {train_std:.4f}")
        print(f"  Val    Balanced Accuracy : {val_mean:.4f} ± {val_std:.4f}")
        print(f"  Test   Balanced Accuracy : {bal_acc_test:.4f}")
        print(f"  Test   Balanced Accuracy by subject     : {test_mean:.4f} ± {test_std:.4f}")
        print(f"  Test   Macro Precision   : {macro['precision']:.4f}")
        print(f"  Test   Macro Recall      : {macro['recall']:.4f}")
        print(f"  Test   Macro F1          : {macro['f1-score']:.4f}")
        print(f"\n  Classification report completo:")
        print(classification_report(y_eval, y_pred_ev, target_names=[str(c) for c in classes],
                                    zero_division=0))

        # ── AUC (binario: usa columna clase 1; multiclase: macro OvR) ─────────
        auc_val = np.nan
        fpr_arr = tpr_arr = None
        if y_score is not None:
            try:
                if len(classes) == 2:
                    fpr_arr, tpr_arr, _ = roc_curve(y_eval, y_score_ev[:, 1])
                    auc_val = auc(fpr_arr, tpr_arr)
                else:
                    # multiclase: macro OvR
                    from sklearn.preprocessing import label_binarize
                    y_bin  = label_binarize(y_eval, classes=list(range(len(classes))))
                    aucs   = []
                    fpr_all, tpr_all = [], []
                    for ci in range(len(classes)):
                        if y_bin[:, ci].sum() == 0:
                            continue
                        fci, tci, _ = roc_curve(y_bin[:, ci], y_score_ev[:, ci])
                        aucs.append(auc(fci, tci))
                        fpr_all.append(fci); tpr_all.append(tci)
                    if aucs:
                        auc_val = np.mean(aucs)
                        # Para la figura conjunta, usamos la primera clase (representativa)
                        fpr_arr, tpr_arr = fpr_all[0], tpr_all[0]
            except Exception as e:
                print(f"  ⚠️  No se pudo calcular AUC: {e}")

        print(f"  Test   AUC               : {auc_val:.4f}" if not np.isnan(auc_val)
              else "  Test   AUC               : N/A")

        roc_data[clf_name] = (fpr_arr, tpr_arr, auc_val) if fpr_arr is not None else None

        # ── Guardar confusion matrix individual ────────────────────────────────
        cm_path = os.path.join(img_dir, f"CM_{clf_name.replace(' ', '_')}.png")
        save_confusion_matrix(y_eval, y_pred_ev, classes,
                              title=f"{clf_name}\nTest — {experiment_name}",
                              filepath=cm_path)

        # ── Acumular métricas ──────────────────────────────────────────────────
        all_metrics[clf_name] = {
            "Train Bal.Acc (mean±std)": f"{train_mean:.4f} ± {train_std:.4f}",
            "Val Bal.Acc (mean±std)":   f"{val_mean:.4f} ± {val_std:.4f}",
            "Test Bal.Acc":             round(bal_acc_test, 4),
            "Macro Precision":          round(macro["precision"], 4),
            "Macro Recall":             round(macro["recall"], 4),
            "Macro F1":                 round(macro["f1-score"], 4),
            "AUC-ROC":                  round(auc_val, 4) if not np.isnan(auc_val) else "N/A",
        }

    # ── CSV fold a fold ───────────────────────────────────────────────────────
    folds_df  = pd.DataFrame(fold_rows)
    fold_path = os.path.join(output_dir,
                             f"folds_{experiment_name.replace(' ', '_')}.csv")
    folds_df.to_csv(fold_path, index=False)
    print(f"\n  💾 Resultados por fold -> {fold_path}")

    # ── CSV resumen (media +- sd) ─────────────────────────────────────────────
    results_df = pd.DataFrame(all_metrics).T
    results_df.index.name = "Model"
    summary_path = os.path.join(output_dir,
                                f"summary_{experiment_name.replace(' ', '_')}.csv")
    results_df.to_csv(summary_path)
    print(f"  💾 Resumen (media+-sd) -> {summary_path}")

    # ── Figura ROC conjunta ────────────────────────────────────────────────────
    roc_path = os.path.join(img_dir, "ROC_conjunta.png")
    plot_all_rocs(roc_data, experiment_name, roc_path)

    print(f"\n{'='*70}")
    print(f"  RESUMEN FINAL — {experiment_name}")
    print(f"{'='*70}")
    print(results_df.to_string())

    return results_df

 Ahora predecimos

In [ ]:
import warnings
warnings.filterwarnings("ignore")

output_dir_lopo = f"{output_path}/LOPO_analysis"


results_bed_lopo = run_lopo_per_model(
    X=X_bed, y=y_bed, groups=g_bed,
    classes=bed_classes,
    experiment_name="Bedford LOPO",
    output_dir=output_dir_lopo,
    k_features=15, # con 15 es suficiente
)

In [ ]:
# Leer el CSV de folds
folds_df = pd.read_csv(f"{output_path}/LOPO_analysis/folds_Bedford_LOPO.csv")

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="test_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="test_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.33, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (test)")
ax.set_title("Test Accuracy by Fold — Bedford LOPO")
ax.legend()
plt.tight_layout()
plt.show()

### 6.2 Predict task

Now lets see if our classifiers perform better on trying to predict the difficulty of the task. First we need to prepare our data, now the labels to predict are the n-back segment, however we will visualize the reuslts in pairwise comparisons

In [ ]:
def prepare_binary(df_all, n_back_a, n_back_b):
    meta_cols = ["participant_id", "epoch_index", "difficulty", "segment_idx",
                 "bedford_score", "bedford_label", "bedford_encoded",
                 "correct_ratio", "n_back"]
    X_cols = [c for c in df_all.columns if c not in meta_cols]
    mask   = df_all["n_back"].isin([n_back_a, n_back_b])
    df_sub = df_all[mask].copy().reset_index(drop=True)
    df_sub["binary_label"] = (df_sub["n_back"] == n_back_b).astype(int)

    X_norm = df_sub[X_cols].values.astype(float)
    for pid in df_sub["participant_id"].unique():
        idx = df_sub["participant_id"] == pid
        X_norm[idx] = StandardScaler().fit_transform(X_norm[idx])

    y      = df_sub["binary_label"].values
    groups = df_sub["participant_id"].values
    nan_mask          = ~np.isnan(X_norm).any(axis=1)
    X_norm, y, groups = X_norm[nan_mask], y[nan_mask], groups[nan_mask]
    print(f"  {n_back_a}-back vs {n_back_b}-back → {nan_mask.sum()} epochs  "
          f"| clase 0: {(y==0).sum()}  clase 1: {(y==1).sum()}")
    return X_norm, y, groups, X_cols

Same as before but now the labels are the n-back

**1-back vs 3-back + LOPO** 

In [ ]:
import warnings
warnings.filterwarnings("ignore")


X_1v3, y_1v3, g_1v3, cols_1v3 = prepare_binary(df_all, 1, 3)

results_1v3_lopo = run_lopo_per_model(
    X=X_1v3, y=y_1v3, groups=g_1v3,
    classes=["1-back", "3-back"],
    experiment_name="Nback 1vs3 LOPO",
    output_dir=output_dir_lopo,
    k_features=15,
)


In [ ]:
# Leer el CSV de folds
folds_df = pd.read_csv(f"{output_path}/LOPO_analysis/folds_Nback_1vs3_LOPO.csv")

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="test_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="test_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (test)")
ax.set_title("Test Accuracy by Fold — 1vs3-back LOPO")
ax.legend()
plt.tight_layout()
plt.show()

**1-back vs 2-back + LOPO** 

In [ ]:
X_1v2, y_1v2, g_1v2, cols_1v2 = prepare_binary(df_all, 1, 2)

results_1v2_lopo = run_lopo_per_model(
    X=X_1v2, y=y_1v2, groups=g_1v2,
    classes=["1-back", "2-back"],
    experiment_name="Nback 1vs2 LOPO",
    output_dir=output_dir_lopo,
    k_features=15,
)



In [ ]:
# Leer el CSV de folds
folds_df = pd.read_csv(f"{output_path}/LOPO_analysis/folds_Nback_1vs2_LOPO.csv")

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="test_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="test_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (test)")
ax.set_title("Test Accuracy by Fold — 1vs2-back LOPO")
ax.legend()
plt.tight_layout()
plt.show()

### 6.3. Intra-participant model

Given the poor results of our classifiers, we can clearly see that the models are not being able to properly classify the subjective mental workload givne processed EEG data in a inter-participant generably way. One reason for this might be due t the enourmous variability in EEG metrics between participants.

Another aproach to see whether our classifiers work better within subject is perform intra participant analysis, to see if we can predict the mental workload with unseen epochs.

--CAMBIOS NUEVOS: HAcer LOPO dentro de cada sujeto

In [ ]:
# Prepare the functions for intra-participant analysis
def pivot_channels_to_features(df_channel, feat_base=FEAT_BASE, extra_cols=None):
    id_cols = ['participant_id', 'epoch_index', 'difficulty', 'n_back']
    if extra_cols:
        id_cols += [c for c in extra_cols if c in df_channel.columns]
    id_cols = [c for c in id_cols if c in df_channel.columns]
    pieces  = []
    for feat in feat_base:
        if feat not in df_channel.columns:
            continue
        p = df_channel.pivot_table(index=id_cols, columns='Channel',
                                   values=feat, aggfunc='mean')
        p.columns = [f"{feat}_{ch}" for ch in p.columns]
        pieces.append(p)
    return pd.concat(pieces, axis=1).reset_index()


# ──────────────────────────────────────────────────────────────────────────
# FIX (revision de codigo, ver chat): el inner GridSearchCV usaba
# StratifiedKFold sobre epochs, lo que volvia a mezclar epochs del MISMO
# segmento entre train y validacion interna -> leakage reintroducido al
# elegir hiperparametros, aunque el fold externo (LOGO por segmento) ya
# estaba bien. Ahora el inner CV tambien agrupa por segmento, usando
# StratifiedGroupKFold (respeta grupos Y intenta preservar balance de
# clases). Si por pocos segmentos en train no se puede hacer ninguna
# particion valida, se cae a un fit sin tuning (sin grid) en vez de crashear.
# ──────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import StratifiedGroupKFold  # (mover tambien arriba, a la celda de imports, si quieres dejarlo todo junto)


def run_within_per_model(df_wide, label_col, classes,
                         experiment_name, output_dir,
                         n_splits=5, k_features=20, selection='anova',
                         class_pair=None, is_bedford=False):
    """
    Clasificacion within-subject con evaluacion separada por modelo.

    Para cada clasificador:
      - Itera sobre participantes (LeaveOneGroupOut intra-sujeto,
        groups = segment_idx -> 9 ciclos: 1 segmento test, 8 train)
      - Inner CV (GridSearchCV) TAMBIEN agrupado por segmento
        (StratifiedGroupKFold sobre los segmentos de train), para no
        reintroducir leakage al elegir hiperparametros.
      - Train acc: media de los folds train
      - Val acc: inner CV agrupado (GridSearchCV)
      - Confusion matrix y ROC individualmente
      - Tabla resumen + ROC conjunta al final

    Args:
        label_col   : columna con las etiquetas (p.ej. 'n_back' o 'bedford_binary')
        classes     : lista de nombres de clase para la confusion matrix
        class_pair  : (a, b) para n-back binario; None si ya esta en label_col como 0/1
        is_bedford  : si True, binariza por mediana intra-sujeto
    """
    img_dir = os.path.join(output_dir,  experiment_name.replace(" ", "_"))
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  WITHIN-SUBJECT (por modelo) — {experiment_name}")
    print(f"{'='*70}")

    classifiers   = get_classifiers()
    meta          = {'participant_id', 'epoch_index', 'difficulty', 'n_back',
                     'bedford_score', 'segment_idx', label_col}
    feat_cols_all = [c for c in df_wide.columns if c not in meta]

    all_metrics = {}
    roc_data    = {}
    fold_rows   = []   # una fila por (modelo, participante, fold)
    oof_rows    = []   # una fila por (modelo, participante, epoch) — predicciones OOF

    min_per_class = max(n_splits, 5)

    for clf_name, cfg in classifiers.items():
        print(f"\n{'─'*60}")
        print(f"  ▶  {clf_name}")
        print(f"{'─'*60}")

        # Acumuladores globales (todos los sujetos concatenados)
        all_y_true, all_y_pred, all_y_score = [], [], []
        # Acumuladores de metricas por fold/sujeto
        train_accs_all, val_accs_all, test_accs_all = [], [], []
        # Mejores hiperparametros acumulados
        best_params_all = []

        # Diagnosticos del fix (para documentar en la memoria)
        outer_total_folds        = 0
        outer_single_class_folds = 0   # folds externos cuyo test es de una sola clase
        n_innercv_fallback       = 0   # veces que no se pudo hacer grid search agrupado

        participants_used = 0

        for pid in sorted(df_wide['participant_id'].unique()):
            sub = df_wide[df_wide['participant_id'] == pid].copy()

            # ── Construir y ───────────────────────────────────────────────────
            if is_bedford:
                # Umbrales FIJOS: LOW<=2 (0), HIGH>=5 (1), MEDIUM=3-4 excluido
                sub = sub[sub[label_col].isin([1, 2, 6, 7, 8, 9, 10])].copy()   # excluye 3,4,5
                y   = (sub[label_col].values >= 6).astype(int)
            elif class_pair is not None:
                a, b = class_pair
                sub  = sub[sub[label_col].isin([a, b])].copy()
                y    = (sub[label_col].values == b).astype(int)
            else:
                y = sub[label_col].values.astype(int)

            valid = [c for c in feat_cols_all if c in sub.columns
                     and not sub[c].isna().any()]
            X = sub[valid].values.astype(float)

            # ── Grupos = segmento N-back (9 segmentos por participante) ──────
            # LeaveOneGroupOut: en cada ciclo 1 segmento es test y los otros 8
            # son train. Asi ninguna epoch del mismo segmento (mismo valor
            # Bedford / nivel N-back) aparece a la vez en train y test, lo que
            # evita inflar las metricas por fuga de informacion.
            groups = sub['segment_idx'].values

            # Se necesitan >=2 clases y >=2 grupos (segmentos) con clases validas
            if (len(np.unique(y)) < 2 or
                    np.bincount(y).min() < 2 or
                    len(np.unique(groups)) < 2):
                continue

            k    = min(k_features, X.shape[1])
            logo = LeaveOneGroupOut()

            y_pred_pid  = np.empty_like(y)
            y_score_pid = np.full(len(y), np.nan)

            for tr, te in logo.split(X, y, groups):
                # Saltar folds cuyo train no tenga las 2 clases (no entrenable)
                if len(np.unique(y[tr])) < 2:
                    y_pred_pid[te] = y[tr][0] if len(tr) else 0
                    continue

                outer_total_folds += 1
                if len(np.unique(y[te])) < 2:
                    # Estructural: el segmento de test tiene una unica etiqueta.
                    outer_single_class_folds += 1

                pipe = Pipeline([
                    ('zscore',   StandardScaler()),
                    ('selector', get_selector(selection, k)),
                    ('clf',      cfg["clf"]),
                ])

                # ── Inner CV agrupado por segmento (FIX) ─────────────────────
                groups_tr      = groups[tr]
                n_inner_groups = len(np.unique(groups_tr))

                if cfg["param_grid"] and n_inner_groups >= 2:
                    inner_k = max(2, min(3, n_inner_groups))
                    try:
                        inner = StratifiedGroupKFold(n_splits=inner_k,
                                                     shuffle=True, random_state=42)
                        grid  = GridSearchCV(pipe, cfg["param_grid"], cv=inner,
                                             scoring='accuracy', n_jobs=-1,
                                             error_score=np.nan)
                        grid.fit(X[tr], y[tr], groups=groups_tr)
                        if np.isnan(grid.best_score_):
                            raise ValueError("grid degenerado: todas las "
                                              "particiones internas invalidas")
                        model   = grid.best_estimator_
                        val_acc = grid.best_score_
                        best_params_all.append(grid.best_params_)
                    except Exception:
                        # Fallback robusto: muy pocos segmentos en train para
                        # poder hacer CV interno agrupado -> se entrena sin
                        # tuning (parametros por defecto del clasificador).
                        n_innercv_fallback += 1
                        model   = pipe.fit(X[tr], y[tr])
                        val_acc = np.nan
                        best_params_all.append({})
                else:
                    # Sin grid (clasificador sin param_grid) o muy pocos
                    # segmentos en train para cualquier split agrupado.
                    if n_inner_groups < 2:
                        n_innercv_fallback += 1
                    model   = pipe.fit(X[tr], y[tr])
                    val_acc = np.nan
                    best_params_all.append({})

                train_accs_all.append(accuracy_score(y[tr], model.predict(X[tr])))
                val_accs_all.append(val_acc)

                y_pred_pid[te] = model.predict(X[te])
                fold_test_acc = accuracy_score(y[te], y_pred_pid[te])
                test_accs_all.append(fold_test_acc)
                fold_rows.append({
                    "model":         clf_name,
                    "participant_id": pid,
                    "fold":          len([r for r in fold_rows if r["model"] == clf_name and r["participant_id"] == pid]),
                    "train_bal_acc": round(accuracy_score(y[tr], model.predict(X[tr])), 4),
                    "val_bal_acc":   round(val_acc, 4) if not np.isnan(val_acc) else None,
                    "test_bal_acc":  round(fold_test_acc, 4),
                })

                try:
                    if hasattr(model, "predict_proba"):
                        y_score_pid[te] = model.predict_proba(X[te])[:, 1]
                    elif hasattr(model, "decision_function"):
                        y_score_pid[te] = model.decision_function(X[te])
                except Exception:
                    pass

            
            # ── Volcado out-of-fold por participante (para metricas por sujeto) ──
            # En este punto y_pred_pid / y_score_pid ya estan completos: cada
            # epoch fue predicha por el fold externo en el que su segmento era
            # test. Guardamos crudo (y_true, y_pred, y_score) para luego poder
            # calcular la balanced accuracy AGREGADA por sujeto (no promediando
            # folds, que son de una sola clase).
            for i in range(len(y)):
                oof_rows.append({
                    "model":          clf_name,
                    "participant_id": pid,
                    "y_true":         int(y[i]),
                    "y_pred":         int(y_pred_pid[i]),
                    "y_score":        float(y_score_pid[i]) if not np.isnan(y_score_pid[i]) else np.nan,
                })

            all_y_true.extend(y.tolist())
            all_y_pred.extend(y_pred_pid.tolist())
            all_y_score.extend(y_score_pid.tolist())
            participants_used += 1

        # ── Metricas globales ──────────────────────────────────────────────────
        yt = np.array(all_y_true)
        yp = np.array(all_y_pred)
        ys = np.array(all_y_score)

        bal_acc_test = accuracy_score(yt, yp)
        report       = classification_report(yt, yp, output_dict=True,
                                             zero_division=0,
                                             target_names=[str(c) for c in classes])
        macro = report["macro avg"]

        train_mean = np.nanmean(train_accs_all); train_std = np.nanstd(train_accs_all)
        val_mean   = np.nanmean(val_accs_all);   val_std   = np.nanstd(val_accs_all)
        test_mean  = np.nanmean(test_accs_all);  test_std  = np.nanstd(test_accs_all)

        print(f"\n  Participantes usados: {participants_used}")

        pct_single = 100 * outer_single_class_folds / outer_total_folds if outer_total_folds else float('nan')
        print(f"  Folds externos (LOGO) con test de una sola clase: "
              f"{outer_single_class_folds}/{outer_total_folds} ({pct_single:.1f}%) "
              f"[estructural: el segmento de test es de una sola etiqueta]")
        if n_innercv_fallback:
            print(f"  ⚠️ Inner CV agrupado no aplicable en {n_innercv_fallback} folds "
                  f"externos (muy pocos segmentos en train) -> se uso fit sin tuning.")

        # Hiperparametros mas comunes
        print(f"\n  Hiperparametros mas frecuentes:")
        if best_params_all and any(best_params_all):
            from collections import Counter
            non_empty = [bp for bp in best_params_all if bp]
            if non_empty:
                for pk in non_empty[0].keys():
                    vals_pk     = [bp[pk] for bp in best_params_all if pk in bp]
                    most_common = Counter(vals_pk).most_common(1)[0]
                    print(f"    {pk}: {most_common[0]}  "
                          f"(en {most_common[1]}/{len(best_params_all)} folds)")
        else:
            print("    (sin grid search para este modelo)")

        print(f"\n  Train  Balanced Accuracy : {train_mean:.4f} ± {train_std:.4f}")
        print(f"  Val    Balanced Accuracy : {val_mean:.4f} ± {val_std:.4f}")
        print(f"  Test   Balanced Accuracy : {bal_acc_test:.4f}  "
              f"(media por fold: {test_mean:.4f} ± {test_std:.4f})")
        print(f"  Test   Macro Precision   : {macro['precision']:.4f}")
        print(f"  Test   Macro Recall      : {macro['recall']:.4f}")
        print(f"  Test   Macro F1          : {macro['f1-score']:.4f}")
        print(f"\n  Classification report completo:")
        print(classification_report(yt, yp, target_names=[str(c) for c in classes],
                                    zero_division=0))

        # ── AUC ───────────────────────────────────────────────────────────────
        auc_val = np.nan; fpr_arr = tpr_arr = None
        if not np.isnan(ys).all():
            mask_valid = ~np.isnan(ys)
            try:
                fpr_arr, tpr_arr, _ = roc_curve(yt[mask_valid], ys[mask_valid])
                auc_val = auc(fpr_arr, tpr_arr)
            except Exception as e:
                print(f"  ⚠️  AUC no calculable: {e}")

        print(f"  Test   AUC               : {auc_val:.4f}" if not np.isnan(auc_val)
              else "  Test   AUC               : N/A")

        roc_data[clf_name] = (fpr_arr, tpr_arr, auc_val) if fpr_arr is not None else None

        # ── Confusion matrix ───────────────────────────────────────────────────
        cm_path = os.path.join(img_dir, f"CM_{clf_name.replace(' ', '_')}.png")
        save_confusion_matrix(yt, yp, classes,
                              title=f"{clf_name}\nTest — {experiment_name}",
                              filepath=cm_path)

        # ── Acumular metricas ──────────────────────────────────────────────────
        all_metrics[clf_name] = {
            "Train Bal.Acc (mean±std)": f"{train_mean:.4f} ± {train_std:.4f}",
            "Val Bal.Acc (mean±std)":   f"{val_mean:.4f} ± {val_std:.4f}",
            "Test Bal.Acc":             round(bal_acc_test, 4),
            "Macro Precision":          round(macro["precision"], 4),
            "Macro Recall":             round(macro["recall"], 4),
            "Macro F1":                 round(macro["f1-score"], 4),
            "AUC-ROC":                  round(auc_val, 4) if not np.isnan(auc_val) else "N/A",
            "Single-class test folds (%)": round(pct_single, 1),
        }

    # ── CSV fold a fold ───────────────────────────────────────────────────────
    folds_df  = pd.DataFrame(fold_rows)
    fold_path = os.path.join(output_dir,
                             f"folds_{experiment_name.replace(' ', '_')}.csv")
    folds_df.to_csv(fold_path, index=False)
    print(f"\n  Resultados por fold -> {fold_path}")
     # ── CSV out-of-fold (predicciones crudas por epoch) ───────────────────────
    oof_df   = pd.DataFrame(oof_rows)
    oof_path = os.path.join(output_dir,
                            f"oof_{experiment_name.replace(' ', '_')}.csv")
    oof_df.to_csv(oof_path, index=False)
    print(f"  Predicciones OOF -> {oof_path}")

    # ── CSV resumen (media +- sd) ─────────────────────────────────────────────
    results_df = pd.DataFrame(all_metrics).T
    results_df.index.name = "Model"
    summary_path = os.path.join(output_dir,
                                f"summary_{experiment_name.replace(' ', '_')}.csv")
    results_df.to_csv(summary_path)
    print(f"  Resumen (media+-sd) -> {summary_path}")

    roc_path = os.path.join(img_dir, "ROC_conjunta.png")
    plot_all_rocs(roc_data, experiment_name, roc_path)

    print(f"\n{'='*70}")
    print(f"  RESUMEN FINAL — {experiment_name}")
    print(f"{'='*70}")
    print(results_df.to_string())

    return results_df


predict!

In [ ]:
output_dir_within = f"{output_path}/within_analysis"

features_all_copy = features_all.copy()

features_all_copy['n_back'] = features_all_copy['difficulty'].str.extract(r'(\d)').astype(int)

# Añadir bedford_score Y segment_idx a features_all
# segment_idx es el grupo para el LeaveOneGroupOut intra-sujeto
labels = df_all[['participant_id', 'epoch_index', 'bedford_score', 'segment_idx']].drop_duplicates()
features_all_copy = features_all_copy.merge(labels, on=['participant_id', 'epoch_index'], how='left')

df_wide_bed = pivot_channels_to_features(features_all_copy, extra_cols=['bedford_score', 'segment_idx'])

results_bed_within = run_within_per_model(
    df_wide     = df_wide_bed,
    label_col   = 'bedford_score',
    classes     = ['LOW (≤2)', 'HIGH (≥6)'],
    experiment_name = "Bedford Within",
    output_dir  = output_dir_within,
    n_splits    = 5,
    k_features  = 20,
    selection   = 'anova',
    is_bedford  = True,
)

In [ ]:
# Leer el CSV de folds
folds_df = pd.read_csv(f"{output_path}/within_analysis/folds_Bedford_Within.csv")

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="test_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="test_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (test)")
ax.set_title("Test Accuracy by Fold — Bedford StratifiedKFold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="train_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="train_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (train)")
ax.set_title("Train Accuracy by Fold — Bedford StratifiedKFold")
ax.legend()
plt.tight_layout()
plt.show()

el random forest está un poco overfitteado en alguno de los folds, pero el logistic regression y el LDA y el decision tree no tantooo

Within 1-back vs 3-back

In [ ]:
# Asegurar que segment_idx (grupo del LeaveOneGroupOut) está en features_all_copy
if 'segment_idx' not in features_all_copy.columns:
    seg_labels = df_all[['participant_id', 'epoch_index', 'segment_idx']].drop_duplicates()
    features_all_copy = features_all_copy.merge(
        seg_labels, on=['participant_id', 'epoch_index'], how='left')

df_wide = pivot_channels_to_features(features_all_copy, extra_cols=['segment_idx'])

results_1v3_within = run_within_per_model(
    df_wide     = df_wide,
    label_col   = 'n_back',
    classes     = ['1-back', '3-back'],
    experiment_name = "Nback 1vs3 Within",
    output_dir  = output_dir_within,
    n_splits    = 5,
    k_features  = 20,
    selection   = 'anova',
    class_pair  = (1, 3),
)



In [ ]:
# Leer el CSV de folds
folds_df = pd.read_csv(f"{output_path}/within_analysis/folds_Nback_1vs3_Within.csv")

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="test_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="test_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (test)")
ax.set_title("Test Accuracy by Fold — Bedford StratifiedKFold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="train_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="train_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (train)")
ax.set_title("Train Accuracy by Fold — Bedford StratifiedKFold")
ax.legend()
plt.tight_layout()
plt.show()

Within 2-back vs 1-back

In [ ]:
results_1v2_within = run_within_per_model(
    df_wide     = df_wide,
    label_col   = 'n_back',
    classes     = ['1-back', '2-back'],
    experiment_name = "Nback 1vs2 Within",
    output_dir  = output_dir_within,
    n_splits    = 5,
    k_features  = 20,
    selection   = 'anova',
    class_pair  = (1, 2),
)

In [ ]:
# Leer el CSV de folds
folds_df = pd.read_csv(f"{output_path}/within_analysis/folds_Nback_1vs2_Within.csv")

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="test_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="test_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (test)")
ax.set_title("Test Accuracy by Fold — StratifiedKFold LOPO")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:

# Violin + scatter
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=folds_df, x="model", y="train_bal_acc",
               inner=None, palette="muted", ax=ax)
sns.stripplot(data=folds_df, x="model", y="train_bal_acc",
              color="black", size=3, jitter=True, alpha=0.6, ax=ax)

ax.axhline(0.5, color="gray", linestyle="--", lw=1, label="Chance")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (test)")
ax.set_title("Train Accuracy by Fold — StratifiedKFold LOGO")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── ¿Hay participantes donde el modelo sí funciona? ────────────────
folds = pd.read_csv(f"{output_dir_within}/folds_Bedford_Within.csv") # cambiar esto por el csv folds que quieras
pivot = folds.groupby(['participant_id', 'model'])['test_bal_acc'].mean().unstack()
print("\nBalanced acc por participante:")
print(pivot.round(3))

# Heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0.5, vmin=0.2, vmax=0.8)
plt.title('Test Accuracy per participant per model')
plt.tight_layout()
plt.savefig(f"{output_dir_within}/heatmap_by_participant.png", dpi=150)
plt.show()

In [ ]:
print(pivot.mean().round(3))           # macro por sujeto (será ~0.40, ruidoso, NO comparable a tabla)

In [ ]:
print(f"\n{'='*70}")
print("  COMPARATIVA GLOBAL — todos los experimentos")
print(f"{'='*70}")

for name, df_r in [
    ("Bedford LOPO",        results_bed_lopo),
    ("N-back 1vs3 LOPO",    results_1v3_lopo),
    ("N-back 1vs2 LOPO",    results_1v2_lopo),
    ("Bedford Within",      results_bed_within),
    ("N-back 1vs3 Within",  results_1v3_within),
    ("N-back 1vs2 Within",  results_1v2_within),
]:
    print(f"\n  ── {name} ──")
    cols_show = [c for c in ["Test Bal.Acc", "Macro F1", "AUC-ROC"] if c in df_r.columns]
    print(df_r[cols_show].to_string())
